# Fase 2: ajuste contrastivo de embeddings (LetraDeMédico)

Entrena los 4 modelos del experimento y los evalúa:

| Modelo base | Variante `split` (experimento honesto) | Variante `completo` (para la app) |
|---|---|---|
| ALIA-MrBERT | sin los 12 analitos de test | todos los analitos |
| multilingual-e5-small | sin los 12 analitos de test | todos los analitos |

**Antes de empezar:** *Entorno de ejecución → Cambiar tipo de entorno → GPU* (L4 o A100 con Colab Pro).

Al final se descarga un `.zip` con los modelos y los resultados. Se descomprime en la raíz del proyecto local.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!git clone https://github.com/Valero25/Proyecto_LaLetraDeMedico.git
%cd Proyecto_LaLetraDeMedico
!git log --oneline -1
!pip install -q sentence-transformers faiss-cpu datasets accelerate python-dotenv

## Entrenamiento

Cada corrida imprime la pérdida cada 10 pasos y guarda `entrenamiento.json` con los hiperparámetros y la curva de pérdida.

In [ ]:
EPOCAS, BATCH, MAX_SEQ = 3, 64, 512

for modelo in ["e5-small", "alia"]:
    for variante in ["split", "completo"]:
        print(f"\n===== {modelo} / {variante} =====")
        !python training/rag/ajustar_embeddings.py --modelo {modelo} --variante {variante} --epocas {EPOCAS} --batch {BATCH} --max-seq {MAX_SEQ}

## Evaluación

Compara los modelos base con los ajustados sobre las 505 consultas.

**La fila que importa es `no_vistos` de los modelos `-split`:** son consultas de analitos que el modelo nunca vio al entrenar. Los modelos `-completo` vieron todos los analitos, así que su `no_vistos` no es una medida justa.

In [ ]:
!python evaluacion/evaluar_rag.py --modelos subcadena bm25 e5-small alia bm25+alias e5-small+alias alia+alias e5-small-ajustado-split+alias alia-ajustado-split+alias e5-small-ajustado-completo+alias alia-ajustado-completo+alias

## Descargar resultados

El zip trae `models/rag/*-ajustado-*` y `evaluacion/resultados/fase2_rag.json`. Descomprímelo en la raíz del proyecto local (las rutas ya vienen incluidas).

In [ ]:
!rm -rf models/rag/*-ajustado-*/checkpoints
!zip -qr fase2_ajuste.zip models/rag/*-ajustado-* evaluacion/resultados/fase2_rag.json
!ls -lh fase2_ajuste.zip

from google.colab import files
files.download("fase2_ajuste.zip")